In [ ]:
import json
import random
from pathlib import Path
from collections import defaultdict

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Rectangle

SFPI = Path("../data/SFPI")

SPLIT = "train"  # change to 'val' or 'test'

with open(SFPI / "Annotations" / f"{SPLIT}_annotation.json") as f:
    coco = json.load(f)

# Merge numbered variants into a single display class
MERGE = {
    "armchair": "armchair",
    "bed":      "bed",
    "door1":    "door",  "door2":   "door",
    "sink1":    "sink",  "sink2":   "sink",  "sink3": "sink", "sink4": "sink",
    "sofa1":    "sofa",  "sofa2":   "sofa",
    "table1":   "table", "table2":  "table", "table3": "table",
    "tub":      "tub",
    "window1":  "window","window2": "window",
}
MERGED_CLASSES = ["armchair", "bed", "door", "sink", "sofa", "table", "tub", "window"]

id2img   = {img["id"]: img for img in coco["images"]}
id2cat   = {c["id"]: MERGE[c["name"]] for c in coco["categories"]}

img2anns = defaultdict(list)
for ann in coco["annotations"]:
    img2anns[ann["image_id"]].append(ann)

print(f"Split       : {SPLIT}")
print(f"Images      : {len(id2img)}")
print(f"Annotations : {len(coco['annotations'])}")
print(f"Classes ({len(MERGED_CLASSES)}): {MERGED_CLASSES}")

In [ ]:
# Consistent colour per merged class across all cells
rng = random.Random(42)
PALETTE = {
    name: tuple(rng.randint(60, 230) / 255 for _ in range(3))
    for name in MERGED_CLASSES
}

def draw_annotations(img_id, ax=None):
    meta     = id2img[img_id]
    img_path = SFPI / "Images" / SPLIT / meta["file_name"]
    img      = cv2.imread(str(img_path))
    if img is None:
        raise FileNotFoundError(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    if ax is None:
        _, ax = plt.subplots(figsize=(14, 10))

    ax.imshow(img)
    ax.set_title(f"{meta['file_name']}  ({meta['width']}x{meta['height']})", fontsize=9)
    ax.axis("off")

    seen = set()
    for ann in img2anns[img_id]:
        x, y, w, h = ann["bbox"]
        name  = id2cat[ann["category_id"]]
        color = PALETTE[name]
        ax.add_patch(Rectangle((x, y), w, h,
                               linewidth=1.5, edgecolor=color, facecolor="none"))
        ax.text(x, y - 3, name, fontsize=6, color=color,
                fontweight="bold", clip_on=True)
        seen.add(name)

    legend = [mpatches.Patch(color=PALETTE[n], label=n) for n in sorted(seen)]
    ax.legend(handles=legend, fontsize=6, loc="upper right",
              framealpha=0.7, ncol=2)
    return ax

In [ ]:
# ── Single image viewer ──────────────────────────────────────────────────────
# Set to None for a random pick, or set to a specific int id
IMAGE_ID = None

img_id = IMAGE_ID or random.choice(list(id2img))
print(f"image_id={img_id}  |  annotations={len(img2anns[img_id])}")

draw_annotations(img_id)
plt.tight_layout()
plt.show()

In [ ]:
# ── Grid viewer — N random images ────────────────────────────────────────────
N_COLS = 3
N_ROWS = 3

sample_ids = random.sample(list(id2img), N_COLS * N_ROWS)
fig, axes  = plt.subplots(N_ROWS, N_COLS, figsize=(18, 13))

for ax, img_id in zip(axes.flat, sample_ids):
    draw_annotations(img_id, ax=ax)

plt.suptitle(f"SFPI — {SPLIT} split  (random sample)", fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Dataset statistics ───────────────────────────────────────────────────────
from collections import Counter

counts = Counter(id2cat[ann["category_id"]] for ann in coco["annotations"])
labels, values = zip(*sorted(counts.items(), key=lambda x: -x[1]))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.barh(labels, values, color=[PALETTE[l] for l in labels])
ax.set_xlabel("Annotation count")
ax.set_title(f"Annotations per class — {SPLIT}")
ax.invert_yaxis()
for i, v in enumerate(values):
    ax.text(v + max(values) * 0.005, i, str(v), va="center", fontsize=8)

ax = axes[1]
ax.pie(values, labels=labels, colors=[PALETTE[l] for l in labels],
       autopct="%1.1f%%", textprops={"fontsize": 7}, startangle=140)
ax.set_title("Class distribution")

plt.suptitle(f"SFPI — {SPLIT} split", fontsize=12)
plt.tight_layout()
plt.show()

print(f"\nAnnotations per image (mean): {len(coco['annotations']) / len(id2img):.1f}")

In [ ]:
# ── Browse images by merged class ─────────────────────────────────────────────
TARGET_CLASS = "door"  # one of: armchair bed door sink sofa table tub window
N_SHOW = 6

imgs_with = list({
    ann["image_id"] for ann in coco["annotations"]
    if id2cat[ann["category_id"]] == TARGET_CLASS
})
sample = random.sample(imgs_with, min(N_SHOW, len(imgs_with)))

print(f"'{TARGET_CLASS}' appears in {len(imgs_with)} images")

n_cols = 3
n_rows = -(-len(sample) // n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 5 * n_rows))

for ax, img_id in zip(axes.flat, sample):
    draw_annotations(img_id, ax=ax)
for ax in axes.flat[len(sample):]:
    ax.axis("off")

plt.suptitle(f"Images containing '{TARGET_CLASS}'", fontsize=11)
plt.tight_layout()
plt.show()